# Analisi Conv-E

## Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.hpo import hpo_pipeline
import json

In [3]:
from data_loader import AtravelDataset

# path
BASE_PATH = "C:\\Users\\User\\Documents\\anisa\\Progetto_DL\\atravel-materialize"

# Inizializzazione
dataset = AtravelDataset(BASE_PATH)
data = dataset.get_splits()

# Data
train_edges, train_types = data['train']
print(f"Loaded {train_edges.size(1)} triples for training.")

Loaded 65401 triples for training.


## Triples factory

In [ ]:
def get_pykeen_triples(dataset_split, entity_to_id=None, relation_to_id=None):
    # dataset_split è (edge_index, edge_type)
    edge_index, edge_type = dataset_split
    
    # le triple in formato [N, 3] come stringhe
    triples = torch.stack([
        edge_index[0], # Soggetto
        edge_type,    # Relazione
        edge_index[1]  # Oggetto
    ], dim=1).cpu().numpy().astype(str)
    
    # la TriplesFactory
    return TriplesFactory.from_labeled_triples(
        triples, 
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=False
    )

# Il training factory genera le mappature ID basandosi sui nomi trovati
training_factory = get_pykeen_triples(dataset.get_splits()['train'])

validation_factory = get_pykeen_triples(
    dataset.get_splits()['valid'],
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id
)

testing_factory = get_pykeen_triples(
    dataset.get_splits()['test'],
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id
)

## Tuning degli iperparametri

In [ ]:
hpo_result = hpo_pipeline(
    training=training_factory,
    validation=validation_factory,
    testing=testing_factory,
    model='ConvE',
    model_kwargs_ranges=dict(
        embedding_dim=dict(type=int, low=64, high=128, q=64),
        input_dropout=dict(type=float, low=0.1, high=0.3),
        feature_map_dropout=dict(type=float, low=0.1, high=0.3),
        output_dropout=dict(type=float, low=0.1, high=0.3),
    ),
    model_kwargs=dict(
    ),
    optimizer='Adam',
    optimizer_kwargs_ranges=dict(
        lr=dict(type=float, low=0.001, high=0.01, log=True), 
    ),
    training_kwargs=dict(
        num_epochs=30,
        batch_size=1024,
    ),
    metric='mrr',
    evaluator_kwargs=dict(
        filtered=True, 
    ),
    n_trials=5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

[I 2025-12-31 00:13:55,357] A new study created in memory with name: no-name-5bd07e0d-90ee-4f31-b0ce-3c33142055bc
No random seed is specified. Setting to 1582261106.

The ConvE model should be trained with inverse triples.
This can be done by defining the TriplesFactory class with the _create_inverse_triples_ parameter set to true.
Training epochs on cuda:0: 100%|██████████| 30/30 [00:18<00:00,  1.60epoch/s, loss=0.00169, prev_loss=0.00191]
Evaluating on cuda:0: 100%|██████████| 3.85k/3.85k [03:22<00:00, 19.0triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 227.07s seconds
[I 2025-12-31 00:18:01,750] Trial 0 finished with value: 0.06199226528406143 and parameters: {'model.output_channels': 16, 'model.input_dropout': 0.12214254180911806, 'model.output_dropout': 0.12567419535047475, 'model.feature_map_dropout': 0.20115236134005463, 'model.embedding_dim': 64, 'optimizer.lr': 0.00603080980159535, 'negative_sampler.num_negs_per_pos': 1}. Best is trial 0 with value: 0.06199226528406

In [6]:
study = hpo_result.study

In [7]:
best_trial = study.best_trial
best_trial.value

0.2213592678308487

## Replica finale 
per ottenere le metriche

In [ ]:
best_params = best_trial.params

final_result_conve = pipeline(
    training=training_factory,
    testing=testing_factory,
    validation=validation_factory,
    model='ConvE',
    model_kwargs=dict(
        embedding_dim=best_params['model.embedding_dim'],
        input_dropout=best_params['model.input_dropout'],
        feature_map_dropout=best_params['model.feature_map_dropout'],
        output_dropout=best_params['model.output_dropout'],
    ),
    optimizer='Adam',
    optimizer_kwargs=dict(
        lr=best_params['optimizer.lr'],
    ),
    training_kwargs=dict(
        num_epochs=30,
        batch_size=1024,
    ),
    evaluator='rankbased',
    evaluator_kwargs=dict(
        metrics=['mrr', 'mr', 'hits@k'],
        filtered=True, 
    ),
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

INFO:pykeen.pipeline.api:Using device: cuda
The ConvE model should be trained with inverse triples.
This can be done by defining the TriplesFactory class with the _create_inverse_triples_ parameter set to true.
INFO:pykeen.nn.modules:Resolving None * None * None = 128.
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
Training epochs on cuda:0: 100%|██████████| 30/30 [00:31<00:00,  1.04s/epoch, loss=0.00218, prev_loss=0.00322]
Evaluating on cuda:0: 100%|██████████| 7.70k/7.70k [1:35:56<00:00, 1.34triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 5810.22s seconds


## Risultati

In [ ]:
metric_dict = final_result_conve.metric_results.to_dict()

output_file = "metriche_convE.json"
with open(output_file, "w") as f:
    json.dump(metric_dict, f, indent=4)  

print(f"Metriche salvate in {output_file}")

Metriche salvate in metriche_convE.json


In [10]:
print("MRR:", final_result_conve.get_metric('mrr'))
print("MR:", final_result_conve.get_metric('mr'))
print("Hits@1:", final_result_conve.get_metric('hits@1'))
print("Hits@3:", final_result_conve.get_metric('hits@3'))
print("Hits@10:", final_result_conve.get_metric('hits@10'))

MRR: 0.15999728441238403
MR: 7329.142578125
Hits@1: 0.12547108512020794
Hits@3: 0.17017543859649123
Hits@10: 0.2179337231968811
